<h1 style="text-align: center;"> Data Preprocessing & Feature Engineering </h1>
<h4 style="text-align: center;">  From Raw Data to Model-Ready Features </h4>

## 📘 When and Why to Scale

- **Scale-sensitive models:** KNN, SVM, Linear/Logistic Regression, Neural Nets, PCA, K-means → *must* scale.  
- **Mostly scale-invariant:** Decision Trees, Random Forest, Gradient Boosting, XGBoost → scaling has little effect on splits.  
- **Outliers present?** Prefer **RobustScaler** (median & IQR) or log transforms for skewed features.  
- **Data leakage warning:** Fit scalers **only** on the **training** split. Use **Pipeline** so you don’t forget. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
import plotly.express as px

# Set Float precision
pd.options.display.float_format = '{:,.3f}'.format
# Set visual style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [2]:
df = pd.read_csv('../data/cleaned_data.csv')
df.head()

,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,Brand,Model
0,Mumbai,2010,72000,CNG,Manual,First,11.438,998.000,58.160,5.000,1.750,Maruti,Wagon
1,Pune,2015,41000,Diesel,Manual,First,19.670,"1,582.000",126.200,5.000,12.500,Hyundai,Creta
2,Chennai,2011,46000,Petrol,Manual,First,18.200,"1,199.000",88.700,5.000,4.500,Honda,Jazz
3,Chennai,2012,87000,Diesel,Manual,First,20.770,"1,248.000",88.760,7.000,6.000,Maruti,Ertiga
4,Coimbatore,2013,40670,Diesel,Automatic,Second,15.200,"1,968.000",140.800,5.000,17.740,Audi,A4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6015 entries, 0 to 6014
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Location           6015 non-null   object 
 1   Year               6015 non-null   int64  
 2   Kilometers_Driven  6015 non-null   int64  
 3   Fuel_Type          6015 non-null   object 
 4   Transmission       6015 non-null   object 
 5   Owner_Type         6015 non-null   object 
 6   Mileage            5945 non-null   float64
 7   Engine             5979 non-null   float64
 8   Power              5873 non-null   float64
 9   Seats              5973 non-null   float64
 10  Price              6015 non-null   float64
 11  Brand              6015 non-null   object 
 12  Model              6015 non-null   object 
dtypes: float64(5), int64(2), object(6)
memory usage: 611.0+ KB


In [4]:
df['Seats'] = df['Seats'].astype('category')

In [5]:
df.describe()

,Year,Kilometers_Driven,Mileage,Engine,Power,Price
count,"6,015.000","6,015.000","5,945.000","5,979.000","5,873.000","6,015.000"
mean,"2,013.358","57,670.792",18.195,"1,619.955",113.128,9.425
std,3.270,"37,870.190",4.152,598.896,53.507,10.905
min,"1,998.000",171.000,5.676,72.000,34.200,0.440
25%,"2,011.000","34,000.000",15.150,"1,198.000",75.000,3.500
50%,"2,014.000","53,000.000",18.120,"1,493.000",97.700,5.630
75%,"2,016.000","73,000.000",21.030,"1,984.000",138.100,9.950
max,"2,019.000","775,000.000",28.400,"5,998.000",552.000,100.000


In [6]:
# First, Split the data into train and test sets
from sklearn.model_selection import train_test_split

X = df.drop(['Price'] , axis=1)
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
num_features = X.select_dtypes(include=[np.number]).columns.to_list()
num_features.remove('Year')  # removing 'Year' if present
num_features

['Kilometers_Driven', 'Mileage', 'Engine', 'Power']

# Numerical Features Scaling

### ✅ Min-Max Scaling

Min-Max scaling transforms features to a fixed range, usually [0, 1].  
It preserves the shape of the original distribution but rescales the data.

The formula is:

$$
X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}
$$

Where:
- $X$ is the original feature value  
- $X_{min}$ is the minimum value of the feature  
- $X_{max}$ is the maximum value of the feature  
- $X_{scaled}$ is the scaled value in the range [0,1]


### ✅ Standardization (Z-score Scaling)

Standardization rescales features to have a mean of 0 and a standard deviation of 1.  
It is useful when features have different units or scales.

The formula is:

$$
X_{scaled} = \frac{X - \mu}{\sigma}
$$

Where:
- $X$ is the original feature value  
- $\mu$ is the mean of the feature in the training data  
- $\sigma$ is the standard deviation of the feature in the training data  
- $X_{scaled}$ is the standardized value


### ✅ Robust Scaling

Robust scaling uses statistics that are robust to outliers: the median and the interquartile range (IQR).  
It is especially useful when the data contains outliers.

The formula is:

$$
X_{scaled} = \frac{X - \text{median}}{Q_3 - Q_1}
$$

Where:
- $X$ is the original feature value  
- $\text{median}$ is the median of the feature  
- $Q_1$ and $Q_3$ are the 25th and 75th percentiles, respectively  
- $X_{scaled}$ is the scaled value


In [8]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import pandas as pd

class OutlierHandler(BaseEstimator, TransformerMixin):
    def __init__(self, method='iqr', lower_percentile=25, upper_percentile=75,
                 iqr_factor=1.5, z_thresh=3.0, handle='winsorize'):
        self.method = method
        self.lower_percentile = lower_percentile
        self.upper_percentile = upper_percentile
        self.iqr_factor = iqr_factor
        self.z_thresh = z_thresh
        self.handle = handle
        self.bounds_ = {}
        self.columns_ = None

    def fit(self, X, y=None):
        # Ensure input is a DataFrame
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        self.columns_ = X.select_dtypes(include=np.number).columns

        if self.method == 'iqr':
            Q1 = X[self.columns_].quantile(self.lower_percentile / 100.0)
            Q3 = X[self.columns_].quantile(self.upper_percentile / 100.0)
            IQR = Q3 - Q1
            lower = Q1 - self.iqr_factor * IQR
            upper = Q3 + self.iqr_factor * IQR
        elif self.method == 'zscore':
            mean = X[self.columns_].mean()
            std = X[self.columns_].std()
            lower = mean - self.z_thresh * std
            upper = mean + self.z_thresh * std
        else:
            raise ValueError("method must be 'iqr' or 'zscore'")

        self.bounds_ = {col: (lower[col], upper[col]) for col in self.columns_}
        return self

    def transform(self, X):
        # Convert to DataFrame (keep column names if possible)
        if isinstance(X, np.ndarray):
            if self.columns_ is not None and X.shape[1] == len(self.columns_):
                X = pd.DataFrame(X, columns=self.columns_)
            else:
                X = pd.DataFrame(X)

        X_copy = X.copy()

        # Apply outlier handling
        for col, (lower, upper) in self.bounds_.items():
            if self.handle == 'remove':
                X_copy[col] = X_copy[col].where((X_copy[col] >= lower) & (X_copy[col] <= upper))
            elif self.handle == 'winsorize':
                X_copy[col] = np.clip(X_copy[col], lower, upper)
            else:
                raise ValueError("handle must be 'remove' or 'winsorize'")
        
        return X_copy

    def fit_transform(self, X, y=None, **fit_params):
        return self.fit(X, y).transform(X)

    def get_feature_names_out(self, input_features=None):
        # Return the same feature names
        if input_features is None and hasattr(self, "columns_"):
            return self.columns_
        return input_features


In [9]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Suppose num_features is a list of numeric column names
num_pipeline_with_outlier_handler = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('outlier_handler', OutlierHandler(method='iqr', handle='winsorize')),
    ('scaler', MinMaxScaler())
])

num_pipeline_without_outlier_handler = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

transformer = ColumnTransformer([
    ('num', num_pipeline_with_outlier_handler, num_features),
    ('num2', num_pipeline_without_outlier_handler, ['Year'])
], remainder='drop').set_output(transform='pandas')


In [10]:
X_train_prepared = transformer.fit_transform(X_train)
X_test_prepared = transformer.transform(X_test)

X_train_prepared


,num__Kilometers_Driven,num__Mileage,num__Engine,num__Power,num2__Year
5639,1.000,0.399,0.497,0.308,0.381
2303,0.845,0.468,0.467,0.339,0.381
2617,0.999,0.570,0.434,0.179,0.476
1397,0.275,0.425,0.957,1.000,0.857
3703,0.395,0.569,0.495,0.475,0.714
...,...,...,...,...,...
3772,0.115,0.561,0.369,0.247,0.810
5191,0.356,0.618,0.467,0.282,0.762
5226,0.565,0.736,0.630,0.804,0.667
5390,0.350,0.561,0.369,0.246,0.571


In [ ]:
for col in X_train_prepared.columns:
    px.histogram(X_train_prepared, x=col, marginal='box', title=f'Distribution of {col}').show()

In [12]:
def try_scaler(scaler, X_train, X_test, num_features):
    # Create a pipeline for numerical features
    num_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('outlier_handler', OutlierHandler(method='iqr', handle='winsorize')),
        ('scaler', scaler)
    ])
    
    # Combine pipelines using ColumnTransformer
    transformer = ColumnTransformer(transformers=[
        ('num', num_pipeline, num_features)
    ], remainder='drop', 
    ).set_output(transform='pandas')
    
    X_train_prepared = transformer.fit_transform(X_train)
    X_test_prepared = transformer.transform(X_test)
    
    return X_train_prepared, X_test_prepared

In [13]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

minmax_scaler = MinMaxScaler()
standard_scaler = StandardScaler()
robust_scaler = RobustScaler()

for scaler in [MinMaxScaler(), StandardScaler(), RobustScaler()]:
    X_train_prepared, X_test_prepared = try_scaler(scaler, X_train, X_test, num_features)
    print(f"\nScaler: {scaler.__class__.__name__}")
    print("Train set")
    display(X_train_prepared.describe())
   


Scaler: MinMaxScaler
Train set


,num__Kilometers_Driven,num__Mileage,num__Engine,num__Power
count,"4,812.000","4,812.000","4,812.000","4,812.000"
mean,0.431,0.529,0.503,0.393
std,0.229,0.190,0.185,0.237
min,0.000,0.000,0.000,0.000
25%,0.263,0.393,0.369,0.226
50%,0.410,0.528,0.455,0.307
75%,0.558,0.654,0.621,0.536
max,1.000,1.000,1.000,1.000



Scaler: StandardScaler
Train set


,num__Kilometers_Driven,num__Mileage,num__Engine,num__Power
count,"4,812.000","4,812.000","4,812.000","4,812.000"
mean,-0.000,-0.000,-0.000,0.000
std,1.000,1.000,1.000,1.000
min,-1.882,-2.783,-2.718,-1.660
25%,-0.735,-0.720,-0.726,-0.707
50%,-0.091,-0.005,-0.257,-0.365
75%,0.553,0.656,0.639,0.600
max,2.485,2.474,2.686,2.559



Scaler: RobustScaler
Train set


,num__Kilometers_Driven,num__Mileage,num__Engine,num__Power
count,"4,812.000","4,812.000","4,812.000","4,812.000"
mean,0.071,0.004,0.188,0.280
std,0.776,0.727,0.733,0.765
min,-1.390,-2.019,-1.803,-0.991
25%,-0.500,-0.519,-0.344,-0.262
50%,0.000,0.000,0.000,0.000
75%,0.500,0.481,0.656,0.738
max,2.000,1.802,2.156,2.238


In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score


for scaler in [MinMaxScaler(), StandardScaler(), RobustScaler()]:
    X_train_prepared, X_test_prepared = try_scaler(scaler, X_train, X_test, num_features)
    lin_reg = LinearRegression()
    scores = cross_val_score(lin_reg, X_train_prepared, y_train, scoring='neg_mean_squared_error', cv=5)
    rmse_scores = np.sqrt(-scores)
    print(f"\nScaler: {scaler.__class__.__name__}")
    print("Mean RMSE:", rmse_scores.mean())


Scaler: MinMaxScaler
Mean RMSE: 6.645237825907984

Scaler: StandardScaler
Mean RMSE: 6.645237825907984

Scaler: RobustScaler
Mean RMSE: 6.645237825907984
